In [0]:
%run "../includes/configuration"

In [0]:
from pyspark.sql.functions import lit, col, current_timestamp

In [0]:
movie_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/movies")
)

In [0]:
countries_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/countries")
)

In [0]:
production_countries_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/production_countries")
)

In [0]:
movie_companies_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/movies_companies")
)

In [0]:
production_companies_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/production_companies")
)

In [0]:
results_country_prod_company = (
    movie_df.filter(movie_df["year_release_date"] > 2010)
        .join(production_countries_df,
              movie_df["movie_id"] == production_countries_df["movie_id"], "inner")
        .join(countries_df,
              production_countries_df["country_id"] == countries_df["country_id"], "inner")
        .join(movie_companies_df,
              movie_df["movie_id"] == movie_companies_df["movie_id"], "inner")
        .join(production_companies_df,
              movie_companies_df["company_id"] == production_companies_df["company_id"], "inner")
).select(movie_df["title"], movie_df["budget"], movie_df["revenue"],
         movie_df["duration_time"],movie_df["release_date"],
         countries_df["country_name"],
         production_companies_df["company_name"]
).withColumn("created_date", current_timestamp())

DataFrame[title: string, budget: double, revenue: double, duration_time: int, release_date: date, country_name: string, company_name: string, created_date: timestamp]

In [0]:
display(results_country_prod_company)

title,budget,revenue,duration_time,release_date,country_name,company_name
Sin City: A Dame to Kill For,6.5E7,3.9407616E7,102,2014-08-20,United States of America,915
The Amazing Spider-Man,2.15E8,7.52215857E8,136,2012-06-27,United States of America,59
The Ides of March,1.25E7,7.5993061E7,101,2011-09-24,United States of America,306
Sin City: A Dame to Kill For,6.5E7,3.9407616E7,102,2014-08-20,United States of America,2721
The Amazing Spider-Man,2.15E8,7.52215857E8,136,2012-06-27,United States of America,12
The Tree of Life,3.2E7,5.4674226E7,139,2011-05-18,United States of America,8411
The Ides of March,1.25E7,7.5993061E7,101,2011-09-24,United States of America,2124
Sin City: A Dame to Kill For,6.5E7,3.9407616E7,102,2014-08-20,United States of America,4258
The Amazing Spider-Man,2.15E8,7.52215857E8,136,2012-06-27,United States of America,14
The Ides of March,1.25E7,7.5993061E7,101,2011-09-24,United States of America,1885


In [0]:
(
    results_country_prod_company
    .write
        .mode("overwrite")
        .format("parquet")
        .save(f"{gold_folder_path}/results_country_prod_company")
)